# KSPEC Commissioning — Reduced Spectra

Plots every `_red.fits` produced by `scripts/reduce_comm_batch.py`.

- One figure per tile file, one panel per fiber
- Fibers filtered by `TYPE`: **P** (program), **S** (sky), **C** (calibration)
- Spectral annotations: emission lines (blue), absorption lines (red), sky lines (green dotted), telluric bands (green fill)

In [1]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = 'retina'

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

from kspecdr.utils.plot import plot_spectrum

## Configuration

In [3]:
WD   = Path.home() / "Research/kspec/kspecdr"
COMM = WD / "resources" / "comm"

DATES = [
    "20260124",
    "20260125",
    "20260127",
    "20260128",
    "20260129",
    "20260204",
    "20260205",
]

# Fiber types to plot (P = program, S = sky, C = calibration, N = unassigned)
PLOT_TYPES = ["P", "S", "C"]

# Layout
NCOLS         = 2
FIGSIZE_PANEL = (10, 3)   # width × height per panel

# Y-axis limits (None = auto per fiber)
YLIM = (-50, 600)

## Collect all reduced files

In [4]:
red_files: dict[str, list[Path]] = {}   # date -> sorted list of _red.fits

for date in DATES:
    proc_dir = COMM / date / "processed"
    files = sorted(proc_dir.glob("ctile_*_red.fits")) if proc_dir.exists() else []
    red_files[date] = files
    print(f"{date}: {len(files)} file(s)")

total = sum(len(v) for v in red_files.values())
print(f"\nTotal: {total} reduced file(s)")

20260124: 12 file(s)
20260125: 6 file(s)
20260127: 9 file(s)
20260128: 17 file(s)
20260129: 13 file(s)
20260204: 22 file(s)
20260205: 0 file(s)

Total: 79 reduced file(s)


## Helper: read fiber info from FIBRES extension

In [5]:
def read_fiber_info(hdul: fits.HDUList):
    """
    Return (names, types) arrays from the FIBRES BinTable, or
    (None, None) if the extension is absent.
    """
    for hdu in hdul:
        if "FIBRES" in hdu.name:
            tbl   = hdu.data
            names = tbl["NAME"].astype(str) if "NAME" in tbl.names else None
            types = tbl["TYPE"].astype(str) if "TYPE" in tbl.names else None
            return names, types
    return None, None


def read_spectra(red_path: Path):
    """
    Load wave, flux, rwss, fiber names, fiber types from a _red.fits file.
    Returns wave (nfib × npix or 1-D), flux (nfib × npix),
    rwss (nfib × npix or None), names, types.
    rwss contains non-sky-subtracted spectra (RWSS extension), or None if absent.
    """
    with fits.open(red_path) as hdul:
        flux      = hdul["PRIMARY"].data.astype(float)
        wave_data = hdul["WAVELA"].data.astype(float)
        rwss      = hdul["RWSS"].data.astype(float) if "RWSS" in hdul else None
        names, types = read_fiber_info(hdul)

    nfib = flux.shape[0]
    if wave_data.ndim == 1:
        wave = np.tile(wave_data, (nfib, 1))
    else:
        wave = wave_data

    if names is None:
        names = np.array([f"fiber {i}" for i in range(nfib)])
    if types is None:
        types = np.array(["N"] * nfib)

    return wave, flux, rwss, names, types

## Plot all reduced files

One figure per tile, one panel per selected fiber.

Set `target_z` below to apply a redshift to the line annotations (or override per-object in the loop).

In [6]:
# Default redshift for line annotations (0 = observer frame)
target_z = 0.0

for date in DATES:
    files = red_files[date]
    if not files:
        continue

    for red_path in files:
        wave_all, flux_all, rwss_all, names, types = read_spectra(red_path)

        # Select fibers to plot
        sel_idx = [i for i, t in enumerate(types) if t in PLOT_TYPES]
        if not sel_idx:
            print(f"  {red_path.name}: no fibers with type in {PLOT_TYPES}")
            continue

        # Compute shared ylim for this tile from all selected fibers
        # Sky fibers use RWSS (non-sky-subtracted); others use sky-subtracted flux
        def _fiber_flux(i):
            if types[i] == "S" and rwss_all is not None:
                return rwss_all[i]
            return flux_all[i]

        all_flux = np.concatenate([_fiber_flux(i) for i in sel_idx])
        finite = all_flux[np.isfinite(all_flux)]
        if finite.size:
            lo = np.nanpercentile(finite, 1)
            hi = np.nanpercentile(finite, 99)
            margin = 0.1 * (hi - lo) if hi > lo else 50
            tile_ylim = (lo - margin, hi + margin)
        else:
            tile_ylim = None

        nrows = int(np.ceil(len(sel_idx) / NCOLS))
        fig, axes = plt.subplots(
            nrows, NCOLS,
            figsize=(FIGSIZE_PANEL[0] * NCOLS, FIGSIZE_PANEL[1] * nrows),
            constrained_layout=True,
        )
        axes = np.atleast_2d(axes)
        fig.suptitle(f"{date}  —  {red_path.name}", fontsize=10)

        for plot_idx, fib_idx in enumerate(sel_idx):
            row, col = divmod(plot_idx, NCOLS)
            ax = axes[row, col]

            wave = wave_all[fib_idx]
            ftype = types[fib_idx]
            fname = names[fib_idx]

            # Use non-sky-subtracted spectrum for sky fibers
            if ftype == "S" and rwss_all is not None:
                flux = rwss_all[fib_idx]
            else:
                flux = flux_all[fib_idx]

            plot_spectrum(
                wave, flux,
                ax=ax,
                target_z=target_z,
                ylim=tile_ylim,
                title=f"{fname}  [{ftype}]",
            )

        # Hide unused axes
        for plot_idx in range(len(sel_idx), nrows * NCOLS):
            row, col = divmod(plot_idx, NCOLS)
            axes[row, col].set_visible(False)

        fdir = COMM / date / "processed" / "chkimg"
        fdir.mkdir(parents=True, exist_ok=True)
        fpath = fdir / (red_path.stem + ".png")
        fig.savefig(fpath, dpi=300)
        plt.close()